# Pipeline consolidado · AGORA2 → Metage2Metabo (M2M)
## Comunidad mínima para la biotransformación levodopa → dopamina → m-tiramina

**Proyecto de curso — versión reproducible del análisis.** Este notebook consolida y corrige el flujo diagnóstico desarrollado para *Enterococcus faecalis* ATCC 29212 y *Eggerthella lenta* DSM 2243, y permite ampliarlo a los **14 GEMs** preseleccionados.

**Hipótesis biológica:** `levdopa → dpmn → mtym`. **Target computacional:** `M_mtym_e`.

### Reglas de interpretación
- Los modelos de entrada son los **AGORA2 previamente limpiados** (`sbml_sin_ex/`), sin las reacciones de frontera `EX_`, `DM_`, `SK_` y `sink_`. Este notebook **no repone** esas fuentes artificiales y **no altera** los modelos originales.
- En SBML se completan exclusivamente los nombres `name` faltantes. Esto evita que MiSCoTo descarte silenciosamente las bacterias. Se verifica que las redes no cambien.
- `HYP_LEVDOPA_IMPORT: levdopa_e → levdopa_c` es **hipotética y exclusivamente diagnóstica**. No constituye evidencia de un transportador real.
- Añadir `M_dpmn_e` o `M_dpmn_c` al seed es un control positivo, **no** un medio de simulación defendible para demostrar cooperación.
- La esencialidad que devuelve `mincom` es **relativa al conjunto de modelos y supuestos evaluados**. No demuestra esencialidad ecológica ni crecimiento, y no sustituye la validación con FBA/SteadyCom.

**Estructura requerida del proyecto** (el notebook puede ejecutarse desde la raíz o `notebooks/`):

```text
proyecto_redes/
├── sbml_sin_ex/
│   ├── Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml
│   ├── Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml
│   └── ...otros GEMs limpios...
├── m2m/seeds.sbml
├── m2m_targets/seeds.sbml
└── notebooks/Pipeline_oficial_AGORA2_M2M_levodopa.ipynb
```

Resultados y registros: `proyecto_redes/diagnostico_levodopa_oficial/`.

## 0. Comprobar entorno

Usa el kernel donde `m2m` y `clingo` están instalados. El pipeline se ha ejecutado previamente con M2M 1.6.3 / MiSCoTo 3.2.0 en el entorno `opti_redes`; **Python 3.14 no está necesariamente validado por los desarrolladores**. Si reaparecen fallos de interoperabilidad, repetir en Python 3.11.

In [24]:
import csv
import hashlib
import importlib.metadata as im
import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import cobra
from cobra.io import read_sbml_model, write_sbml_model
from lxml import etree
from miscoto import sbml as miscoto_sbml

print("Python:", sys.version.split()[0], sys.executable)
print("m2m:", shutil.which("m2m"))
print("clingo:", shutil.which("clingo"))
for package in ("Metage2Metabo", "miscoto", "menetools", "cobra", "clyngor"):
    try:
        print(f"{package}: {im.version(package)}")
    except im.PackageNotFoundError:
        print(f"{package}: no encontrado")

if not shutil.which("m2m") or not shutil.which("clingo"):
    raise RuntimeError("El kernel actual no encuentra m2m y/o clingo. Selecciona el entorno Conda correcto en VS Code.")

Python: 3.14.7 /Users/javigajardo/miniforge3/envs/opti_redes/bin/python
m2m: /Users/javigajardo/miniforge3/envs/opti_redes/bin/m2m
clingo: /Users/javigajardo/miniforge3/envs/opti_redes/bin/clingo
Metage2Metabo: 1.6.3
miscoto: 3.2.0
menetools: 3.4.0
cobra: 0.32.1
clyngor: 1.0.0


## 1. Configuración y validación de entradas

Se localiza automáticamente la raíz del proyecto. **No cambia rutas globales ni archivos originales.**

In [25]:
def encontrar_raiz():
    cwd = Path.cwd().resolve()
    opciones = [cwd, *cwd.parents, Path.home() / "Desktop/proyecto_redes"]
    for p in opciones:
        if (p / "sbml_sin_ex").is_dir() and (p / "m2m/seeds.sbml").is_file():
            return p.resolve()
    raise FileNotFoundError(
        f"No se encuentra la raíz del proyecto desde {cwd}. "
        "Debe contener sbml_sin_ex/ y m2m/seeds.sbml."
    )

BASE = encontrar_raiz()
SRC = BASE / "sbml_sin_ex"
SEEDS = BASE / "m2m" / "seeds.sbml"
TARGETS = BASE / "m2m_targets" / "seeds.sbml"
WORK = BASE / "diagnostico_levodopa_oficial"
MODELS_OUT = WORK / "modelos"
RUNS_OUT = WORK / "runs"
SEEDS_OUT = WORK / "seeds_control"
LOGS_OUT = WORK / "logs"
for p in [WORK, MODELS_OUT, RUNS_OUT, SEEDS_OUT, LOGS_OUT]:
    p.mkdir(parents=True, exist_ok=True)

EFA = "Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml"
ELENTA = "Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml"
TARGET_ID = "M_mtym_e"
HYP_ID = "HYP_LEVDOPA_IMPORT"

for p in (SRC / EFA, SRC / ELENTA, SEEDS, TARGETS):
    if not p.is_file():
        raise FileNotFoundError(f"Falta un archivo necesario: {p}")

def especies_sbml(path):
    tree = etree.parse(str(path))
    return {x.get("id") for x in tree.xpath("//*[local-name()='listOfSpecies']/*[local-name()='species']")}

seeds_base = especies_sbml(SEEDS)
targets_ids = especies_sbml(TARGETS)
assert "M_levdopa_e" in seeds_base, "El seed original no contiene M_levdopa_e"
assert TARGET_ID in targets_ids, f"El archivo de targets no contiene {TARGET_ID}"
print("Raíz:", BASE)
print("Modelos limpios disponibles:", len(list(SRC.glob("*.xml"))))
print("Número de seeds:", len(seeds_base))
print("Targets:", sorted(targets_ids))
print("Contiene dopamina extracelular:", "M_dpmn_e" in seeds_base)
print("Contiene dopamina citosólica:", "M_dpmn_c" in seeds_base)

Raíz: /Users/javigajardo/Desktop/proyecto_redes
Modelos limpios disponibles: 14
Número de seeds: 104
Targets: ['M_mtym_e']
Contiene dopamina extracelular: False
Contiene dopamina citosólica: False


## 2. Funciones de respaldo, integridad y ejecución

Todo se guarda en una carpeta exclusiva del notebook. Al repetir una celda se pueden reemplazar **solo salidas generadas aquí**, nunca archivos originales de `sbml_sin_ex/`. Cada llamada de M2M conserva un registro de terminal.

In [26]:
def borrar_generado(path):
    path = Path(path)
    raiz = WORK.resolve()
    destino = path.resolve()
    if destino == raiz or not destino.is_relative_to(raiz):
        raise ValueError(f"Se intentó borrar fuera del espacio de trabajo: {path}")
    if path.is_symlink():
        raise ValueError(f"No se permite borrar rutas simbólicas: {path}")
    if path.exists():
        shutil.rmtree(path) if path.is_dir() else path.unlink()


def ejecutar(comando, nombre_log, timeout=3600):
    comando = [str(x) for x in comando]
    print("\n$", " ".join(comando))
    ruta_log = LOGS_OUT / f"{nombre_log}.log"
    try:
        p = subprocess.run(comando, capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired as exc:
        ruta_log.write_text(f"TIMEOUT ({timeout}s): {comando}\n{exc}\n", encoding="utf-8")
        raise
    salida = p.stdout + "\n--- STDERR ---\n" + p.stderr
    ruta_log.write_text("$ " + " ".join(comando) + "\n" + salida, encoding="utf-8")
    print(salida[-2200:])
    if p.returncode:
        raise RuntimeError(f"Error al ejecutar {comando[0]} (código {p.returncode}). Revisa {ruta_log}")
    return p


def huella(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for bloque in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def firma_modelo(model):
    return {
        r.id: (round(r.lower_bound, 9), round(r.upper_bound, 9),
               tuple(sorted((m.id, round(float(x), 9)) for m, x in r.metabolites.items())))
        for r in model.reactions
    }


def reparar_nombres_sbml(origen, destino, verificar=True):
    """Rellena solo name vacío/ausente en species; preserva las redes de reacción."""
    origen, destino = Path(origen), Path(destino)
    arbol = etree.parse(str(origen))
    species = arbol.xpath("//*[local-name()='listOfSpecies']/*[local-name()='species']")
    faltantes = []
    for sp in species:
        if not (sp.get("name") or "").strip():
            if not sp.get("id"):
                raise ValueError(f"Species sin id en {origen.name}")
            sp.set("name", sp.get("id"))
            faltantes.append(sp.get("id"))
    destino.parent.mkdir(parents=True, exist_ok=True)
    arbol.write(str(destino), encoding="UTF-8", xml_declaration=True)

    # Clave: MiSCoTo no debe volver a descartar modelos por metadata SBML.
    try:
        miscoto_sbml.readSBMLnetwork_symbionts_clyngor(str(destino), destino.stem)
    except BaseException as exc:
        raise RuntimeError(f"MiSCoTo no puede leer {destino.name}: {type(exc).__name__}: {exc}") from exc

    if verificar:
        antes = read_sbml_model(str(origen))
        despues = read_sbml_model(str(destino))
        assert firma_modelo(antes) == firma_modelo(despues), f"Se alteraron reacciones: {origen.name}"
        assert {m.id for m in antes.metabolites} == {m.id for m in despues.metabolites}
    return faltantes


def parche_hipotetico(model):
    """Agrega transportador solo en E. faecalis, solo como experimento diagnóstico."""
    if HYP_ID in model.reactions:
        raise ValueError(f"{HYP_ID} ya existe: no se creará por duplicado")
    r = cobra.Reaction(HYP_ID)
    r.name = "Transporte hipotético levodopa e a c (SOLO DIAGNÓSTICO)"
    r.lower_bound, r.upper_bound = 0, 1000
    r.add_metabolites({
        model.metabolites.get_by_id("levdopa_e"): -1,
        model.metabolites.get_by_id("levdopa_c"): 1,
    })
    model.add_reactions([r])
    return r

### Preparar conjuntos bacterianos sin alterar los originales

Variantes disponibles: sin cambios, transportador hipotético y knockouts independientes sobre el escenario hipotético. Se valida **cada SBML con el parser de MiSCoTo** y se comprueba que el número de modelos aceptados coincida con el solicitado.

In [27]:
def preparar_modelos(nombres, etiqueta, agregar_transporte=False, knockout=None):
    """
    knockout: tupla (nombre_archivo, reaction_id). Los knockouts se aplican
    después del transporte hipotético. Todas las transformaciones quedan auditadas.
    """
    nombres = list(nombres)
    if not nombres or len(nombres) != len(set(nombres)):
        raise ValueError("Lista de modelos vacía o con duplicados")
    carpeta = MODELS_OUT / etiqueta
    staging = MODELS_OUT / f"_{etiqueta}_temporal"
    borrar_generado(carpeta)
    borrar_generado(staging)
    carpeta.mkdir(parents=True)
    staging.mkdir(parents=True)
    info = []
    try:
        for nombre in nombres:
            origen = SRC / nombre
            if not origen.is_file():
                raise FileNotFoundError(f"Modelo faltante: {origen}")
            temporal = staging / nombre
            shutil.copy2(origen, temporal)
            cambios = []
            if agregar_transporte and nombre == EFA:
                modelo = read_sbml_model(str(temporal))
                parche_hipotetico(modelo)
                write_sbml_model(modelo, str(temporal))
                cambios.append(HYP_ID)
            if knockout and knockout[0] == nombre:
                modelo = read_sbml_model(str(temporal))
                rid = knockout[1]
                if rid not in modelo.reactions:
                    raise ValueError(f"No existe {rid} en {nombre}")
                modelo.remove_reactions([modelo.reactions.get_by_id(rid)], remove_orphans=False)
                write_sbml_model(modelo, str(temporal))
                cambios.append(f"KO:{rid}")

            faltantes = reparar_nombres_sbml(temporal, carpeta / nombre)
            final = read_sbml_model(str(carpeta / nombre))
            # Comparar contra el modelo original tras aplicar EXACTAMENTE las
            # modificaciones declaradas (y ninguna otra reacción diferente).
            esperado = read_sbml_model(str(origen))
            if agregar_transporte and nombre == EFA:
                parche_hipotetico(esperado)
            if knockout and knockout[0] == nombre:
                rid = knockout[1]
                esperado.remove_reactions(
                    [esperado.reactions.get_by_id(rid)], remove_orphans=False
                )
            if firma_modelo(esperado) != firma_modelo(final):
                raise AssertionError(
                    f"Las reacciones cambiaron más allá de la intervención declarada: {nombre}"
                )
            frontera = [r.id for r in final.reactions
                        if r.id.startswith(("EX_", "DM_", "SK_", "sink_"))]
            if frontera:
                raise ValueError(f"{nombre} contiene aún reacciones frontera: {frontera[:5]}")
            info.append({"archivo": nombre, "se_corrigieron_names": len(faltantes),
                         "ids_reparados": faltantes,
                         "reacciones": len(final.reactions), "cambios": cambios,
                         "sha256_entrada": huella(origen), "sha256_salida": huella(carpeta / nombre)})
    finally:
        borrar_generado(staging)

    if len(list(carpeta.glob("*.xml"))) != len(nombres):
        raise RuntimeError("La carpeta de salida no contiene todos los modelos esperados")
    (carpeta / "_manifest.json").write_text(json.dumps(info, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"{etiqueta}: {len(info)} modelos validados por MiSCoTo")
    for item in info:
        print(f"  {item['archivo']}: {item['reacciones']} reacciones, "
              f"names reparados={item['se_corrigieron_names']}, cambios={item['cambios']}")
    return carpeta


def crear_seed(ids, etiqueta):
    carpeta = SEEDS_OUT / etiqueta
    borrar_generado(carpeta)
    carpeta.mkdir(parents=True)
    ruta_txt = carpeta / "metabolitos.txt"
    ruta_txt.write_text("\n".join(sorted(ids)) + "\n", encoding="utf-8")
    ejecutar(["m2m", "seeds", "--metabolites", ruta_txt, "-o", carpeta], f"seed_{etiqueta}")
    ruta_sbml = carpeta / "seeds.sbml"
    if not ruta_sbml.is_file():
        raise FileNotFoundError(f"M2M no generó {ruta_sbml}")
    if especies_sbml(ruta_sbml) != set(ids):
        raise AssertionError("El seed generado no coincide exactamente con la lista de entrada")
    return ruta_sbml

RESULTADOS = {}

def mincom(modelos, seed, etiqueta, grupo="par", supuesto="ninguno"):
    salida = RUNS_OUT / etiqueta
    borrar_generado(salida)
    ejecutar(["m2m", "mincom", "-n", modelos, "-s", seed,
              "-t", TARGETS, "-o", salida], f"mincom_{etiqueta}")
    json_path = salida / "community_analysis" / "mincom.json"
    if not json_path.is_file():
        raise FileNotFoundError(f"No se generó {json_path}; revisa log de mincom")
    resultado = json.loads(json_path.read_text(encoding="utf-8"))
    RESULTADOS[etiqueta] = {"resultado": resultado, "modelos": str(modelos),
                           "seed": str(seed), "grupo": grupo, "supuesto": supuesto}
    print("\nESCENARIO:", etiqueta)
    print("  Producible:", resultado.get("producible"))
    print("  No producible:", resultado.get("still_unprod"))
    print("  Comunidad mínima reportada:", resultado.get("bacteria"))
    print("  Esenciales:", resultado.get("essential_symbionts"))
    print("  Alternativos:", resultado.get("alternative_symbionts"))
    return resultado

## 3. Control negativo — par original, nombres corregidos

Este escenario usa el seed full original de **104 metabolitos**, sin dopamina añadida ni transportador hipotético. **El bloqueo de m-tiramina es el resultado esperado según las corridas previas**, pero el notebook lo recalcula.

In [28]:
par_original = preparar_modelos([EFA, ELENTA], "par_original_names_corregidos")
resultado_original = mincom(par_original, SEEDS, "01_par_original", supuesto="GEMs limpios; names completados")

par_original_names_corregidos: 2 modelos validados por MiSCoTo
  Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml: 1375 reacciones, names reparados=4, cambios=[]
  Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml: 900 reacciones, names reparados=4, cambios=[]

$ m2m mincom -n /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/modelos/par_original_names_corregidos -s /Users/javigajardo/Desktop/proyecto_redes/m2m/seeds.sbml -t /Users/javigajardo/Desktop/proyecto_redes/m2m_targets/seeds.sbml -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/runs/01_par_original
######### Creating metabolic instance for the whole community #########
Created temporary instance file in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/runs/01_par_original/community_analysis/miscoto_vj1ltnpt.lp

###############################################
#                                             #
#         Minimal community selection         #
#           

## 4. Controles positivos — dopamina como seed

Estas pruebas evalúan si la ruta posterior a dopamina funciona. **No pueden emplearse para inferir cooperación real**: agregar dopamina al seed elimina la necesidad del productor que buscamos identificar.

In [29]:
seed_dpmn_e = crear_seed(seeds_base | {"M_dpmn_e"}, "dopamina_extracelular")
seed_dpmn_c = crear_seed(seeds_base | {"M_dpmn_c"}, "dopamina_citosolica")

resultado_dpmn_e = mincom(
    par_original, seed_dpmn_e, "02_dopamina_extracelular",
    supuesto="M_dpmn_e agregado como control diagnóstico"
)
resultado_dpmn_c = mincom(
    par_original, seed_dpmn_c, "03_dopamina_citosolica",
    supuesto="M_dpmn_c agregado como control diagnóstico"
)


$ m2m seeds --metabolites /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/seeds_control/dopamina_extracelular/metabolitos.txt -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/seeds_control/dopamina_extracelular
Seeds SBML file created in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/seeds_control/dopamina_extracelular/seeds.sbml
--- Total runtime 0.00 seconds ---
--- Logs written in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/seeds_control/dopamina_extracelular/m2m_seeds.log ---

--- STDERR ---


$ m2m seeds --metabolites /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/seeds_control/dopamina_citosolica/metabolitos.txt -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/seeds_control/dopamina_citosolica
Seeds SBML file created in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/seeds_control/dopamina_citosolica/seeds.sbml

## 5. Control positivo condicionado — transporte hipotético de levodopa

Se añade **una sola reacción** a *E. faecalis*:

\[
\mathrm{levdopa_e \rightarrow levdopa_c}
\]

Su propósito es romper la dependencia circular del antiporter `DOPA34DHPHEt` de forma **topológica**. Se conserva el seed original, sin dopamina externa ni citosólica añadida. 

**Resultado ya observado anteriormente:** `M_mtym_e` producible y ambas bacterias seleccionadas y esenciales dentro del par. Esto **no demuestra** que el transportador hipotético sea real.

In [30]:
par_hipotetico = preparar_modelos(
    [EFA, ELENTA], "par_transporte_hipotetico",
    agregar_transporte=True
)
resultado_hipotetico = mincom(
    par_hipotetico, SEEDS, "04_par_transporte_hipotetico",
    supuesto="HYP_LEVDOPA_IMPORT en E. faecalis (NO validado)"
)

assert HYP_ID in read_sbml_model(str(par_hipotetico / EFA)).reactions
if TARGET_ID not in resultado_hipotetico.get("producible", []):
    print("ADVERTENCIA: el target no se produjo; revisar log y reacciones antes de interpretar.")

par_transporte_hipotetico: 2 modelos validados por MiSCoTo
  Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml: 1376 reacciones, names reparados=4, cambios=['HYP_LEVDOPA_IMPORT']
  Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml: 900 reacciones, names reparados=4, cambios=[]

$ m2m mincom -n /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/modelos/par_transporte_hipotetico -s /Users/javigajardo/Desktop/proyecto_redes/m2m/seeds.sbml -t /Users/javigajardo/Desktop/proyecto_redes/m2m_targets/seeds.sbml -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/runs/04_par_transporte_hipotetico
oficial/runs/04_par_transporte_hipotetico/community_analysis/miscoto_vo8a3ki0.lp

###############################################
#                                             #
#         Minimal community selection         #
#                                             #
###############################################

Running minimal community selection
/Users/javi

## 6. Knockouts: dependencia de las reacciones de la cascada

Los tres knockouts se prueban **independientemente** sobre el par con transporte hipotético y seed original:

- `3HLYTCL` en *E. faecalis*: vía de generación de dopamina.
- `DOPADH` en *E. lenta*: reducción de dopamina a m-tiramina.
- `HYP_LEVDOPA_IMPORT` en *E. faecalis*: control de reversión a la condición sin importación hipotética.

Si el target sigue siendo producible después de alguno, **no se debe concluir que el knockout falló**: hay que inspeccionar posibles rutas alternativas o identidades de metabolitos.

In [31]:
KO_DEFINICIONES = [
    ("05_KO_3HLYTCL", EFA, "3HLYTCL"),
    ("06_KO_DOPADH", ELENTA, "DOPADH"),
    ("07_KO_HYP_IMPORT", EFA, HYP_ID),
]

for etiqueta, especie, reaccion in KO_DEFINICIONES:
    carpeta_ko = preparar_modelos(
        [EFA, ELENTA], etiqueta + "_modelos",
        agregar_transporte=True,
        knockout=(especie, reaccion),
    )
    res = mincom(carpeta_ko, SEEDS, etiqueta,
                 supuesto=f"HYP_LEVDOPA_IMPORT + KO {reaccion} en {especie}")
    print("Target tras knockout:", TARGET_ID in res.get("producible", []))

05_KO_3HLYTCL_modelos: 2 modelos validados por MiSCoTo
  Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml: 1375 reacciones, names reparados=4, cambios=['HYP_LEVDOPA_IMPORT', 'KO:3HLYTCL']
  Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml: 900 reacciones, names reparados=4, cambios=[]

$ m2m mincom -n /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/modelos/05_KO_3HLYTCL_modelos -s /Users/javigajardo/Desktop/proyecto_redes/m2m/seeds.sbml -t /Users/javigajardo/Desktop/proyecto_redes/m2m_targets/seeds.sbml -o /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/runs/05_KO_3HLYTCL
######### Creating metabolic instance for the whole community #########
Created temporary instance file in /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/runs/05_KO_3HLYTCL/community_analysis/miscoto_i80ehn1_.lp

###############################################
#                                             #
#         Minimal community selection         

## 7. Ensayos con los 14 GEMs limpios (opcional, ejecutar cuando estés lista)

**Importante:** los 14 modelos deben estar en `sbml_sin_ex/`, incluidos los dos del control. El código verifica uno por uno el parsing MiSCoTo: si alguno falla, detiene la corrida y muestra cuál, en lugar de aceptar una comunidad incompleta.

Se comparan dos escenarios con **idéntico seed full**:

- A. Los GEMs limpios tal como están, completando únicamente los nombres SBML.
- B. El mismo conjunto, agregando el transportador **hipotético** solo a *E. faecalis*.

Si hay 14 modelos, la clasificación de esenciales y alternativos será relativa a esos 14, no solamente al par. Por defecto está desactivado para evitar una corrida extensa accidental.

In [32]:
EJECUTAR_14_MODELOS = True  # Cambia a True cuando quieras analizar la comunidad completa

if EJECUTAR_14_MODELOS:
    todos = [p.name for p in sorted(SRC.glob("*.xml"))]
    if len(todos) != 14:
        print(f"AVISO: hay {len(todos)} SBML, no 14. Revisa la selección de candidatos antes de interpretar.")
    if EFA not in todos or ELENTA not in todos:
        raise ValueError("Los dos GEMs ancla deben estar en la carpeta de 14 modelos")

    todos_originales = preparar_modelos(todos, "14_modelos_originales_corregidos")
    resultado_14_original = mincom(
        todos_originales, SEEDS, "08_14_originales",
        grupo="conjunto_completo", supuesto="14 GEMs limpios; solo metadata name reparada"
    )

    todos_hipoteticos = preparar_modelos(
        todos, "14_modelos_con_transporte_hipotetico", agregar_transporte=True
    )
    resultado_14_hipotetico = mincom(
        todos_hipoteticos, SEEDS, "09_14_transporte_hipotetico",
        grupo="conjunto_completo", supuesto="14 GEMs + HYP_LEVDOPA_IMPORT solo en E. faecalis"
    )
else:
    print("Se omitieron las corridas de 14 modelos. Activa EJECUTAR_14_MODELOS cuando corresponda.")

14_modelos_originales_corregidos: 14 modelos validados por MiSCoTo
  Akkermansia_muciniphila_ATCC_BAA_835_AGORA2_v2.01.xml: 1080 reacciones, names reparados=4, cambios=[]
  Bacteroides_fragilis_638R_AGORA2_v2.01.xml: 1776 reacciones, names reparados=4, cambios=[]
  Bacteroides_thetaiotaomicron_7330_AGORA2_v2.01.xml: 1297 reacciones, names reparados=4, cambios=[]
  Bifidobacterium_longum_DJO10A_AGORA2_v2.01.xml: 1464 reacciones, names reparados=4, cambios=[]
  Eggerthella_lenta_A2_AGORA2_v2.01.xml: 1035 reacciones, names reparados=4, cambios=[]
  Eggerthella_lenta_DSM_11767_AGORA2_v2.01.xml: 1052 reacciones, names reparados=4, cambios=[]
  Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml: 900 reacciones, names reparados=4, cambios=[]
  Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01.xml: 1375 reacciones, names reparados=4, cambios=[]
  Enterococcus_faecium_504_AGORA2_v2.01.xml: 1211 reacciones, names reparados=4, cambios=[]
  Escherichia_coli_042_AGORA2_v2.01.xml: 3028 reacciones, names repara

## 8. Comparación `iscope` vs `cscope` (opcional)

Ejecutar solo si interesa investigar las diferencias previas. **Usar exactamente los mismos modelos y el mismo seed**, porque comparar scopes de archivos `seeds.sbml` distintos produce conclusiones engañosas.

No dependemos aquí de que M2M conserve su `.lp` temporal; utilizamos los JSON que generan las herramientas.

In [33]:
EJECUTAR_SCOPES = False

if EJECUTAR_SCOPES:
    modelos_analisis = par_original  # Cambia a todos_originales si ejecutaste la sección 7
    for comando in ("iscope", "cscope"):
        salida = RUNS_OUT / f"scopes_{comando}"
        borrar_generado(salida)
        cmd = ["m2m", comando, "-n", modelos_analisis, "-s", SEEDS]
        if comando == "cscope":
            cmd += ["-t", TARGETS]
        cmd += ["-o", salida]
        ejecutar(cmd, f"{comando}_mismo_seed")
        print("Salida", comando, [str(x.relative_to(salida)) for x in salida.rglob("*.json")])
else:
    print("Scopes adicionales omitidos (EJECUTAR_SCOPES = False).")

Scopes adicionales omitidos (EJECUTAR_SCOPES = False).


## 9. Resumen reproducible y exportación

Se construye un CSV/JSON a partir **de los resultados realmente obtenidos**, no de valores esperados escritos de antemano. También se guardan los hashes de los seeds y un manifiesto por conjunto de modelos.

In [34]:
columnas = ["escenario", "grupo", "supuesto", "target_producible", "n_bacterias",
            "bacterias", "esenciales", "alternativas", "targets_no_producibles", "modelos", "seed"]
filas = []
for etiqueta, entry in RESULTADOS.items():
    res = entry["resultado"]
    bacterias = res.get("bacteria", []) or []
    if not isinstance(bacterias, list):
        bacterias = [str(bacterias)]
    filas.append({
        "escenario": etiqueta,
        "grupo": entry["grupo"],
        "supuesto": entry["supuesto"],
        "target_producible": TARGET_ID in (res.get("producible") or []),
        "n_bacterias": len(bacterias),
        "bacterias": "; ".join(bacterias),
        "esenciales": "; ".join(res.get("essential_symbionts") or []),
        "alternativas": "; ".join(res.get("alternative_symbionts") or []),
        "targets_no_producibles": "; ".join(res.get("still_unprod") or []),
        "modelos": entry["modelos"],
        "seed": entry["seed"],
    })

csv_path = WORK / "resumen_experimentos.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=columnas)
    w.writeheader()
    w.writerows(filas)

resumen = {
    "proyecto": "Levodopa AGORA2 + M2M",
    "fecha_exportacion": datetime.now().astimezone().isoformat(timespec="seconds"),
    "target": TARGET_ID,
    "seed_original_sha256": huella(SEEDS),
    "targets_sbml_sha256": huella(TARGETS),
    "python": sys.version,
    "m2m_version": im.version("Metage2Metabo"),
    "miscoto_version": im.version("miscoto"),
    "escenarios": filas,
}
json_path = WORK / "resumen_experimentos.json"
json_path.write_text(json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8")

for row in filas:
    print(f"{row['escenario']:38} target={str(row['target_producible']):5} "
          f"n={row['n_bacterias']} | {row['bacterias']}")
print("\nCSV:", csv_path)
print("JSON:", json_path)

01_par_original                        target=False n=0 | 
02_dopamina_extracelular               target=True  n=1 | Eggerthella_lenta_DSM_2243_AGORA2_v2.01
03_dopamina_citosolica                 target=True  n=1 | Eggerthella_lenta_DSM_2243_AGORA2_v2.01
04_par_transporte_hipotetico           target=True  n=2 | Eggerthella_lenta_DSM_2243_AGORA2_v2.01; Enterococcus_faecalis_ATCC_29212_AGORA2_v2.01
05_KO_3HLYTCL                          target=False n=0 | 
06_KO_DOPADH                           target=False n=0 | 
07_KO_HYP_IMPORT                       target=False n=0 | 
08_14_originales                       target=True  n=2 | Escherichia_coli_042_AGORA2_v2.01; Eggerthella_lenta_DSM_11767_AGORA2_v2.01
09_14_transporte_hipotetico            target=True  n=2 | Escherichia_coli_042_AGORA2_v2.01; Eggerthella_lenta_DSM_11767_AGORA2_v2.01

CSV: /Users/javigajardo/Desktop/proyecto_redes/diagnostico_levodopa_oficial/resumen_experimentos.csv
JSON: /Users/javigajardo/Desktop/proyecto_redes/diagn

## 10. Qué se puede concluir y qué queda pendiente

### Observaciones previas que motivaron este notebook
1. Los modelos limpios originales dejaban `M_mtym_e` como no producible con el seed full.
2. Añadir dopamina externa o citosólica al seed hacía producir m-tiramina a *E. lenta* sola.
3. Tras corregir cuatro atributos `name` faltantes por modelo, MiSCoTo dejó de ignorar silenciosamente las bacterias.
4. Con **una importación hipotética de levodopa en *E. faecalis***, el par devolvió `M_mtym_e` producible y las dos bacterias como esenciales **entre esos dos candidatos**.

Estas observaciones se consignan como **resultados de las corridas de diagnóstico anteriores**. Los resultados de este notebook se calculan nuevamente al ejecutar cada sección.

### Antes de afirmar que la comunidad es funcional y crece
- Validar biológicamente el transporte de levodopa, o presentarlo claramente como supuesto de gap-filling. No presentar la reacción hipotética como anotación original de AGORA2.
- Examinar rutas alternativas tras los knockouts. Si algún KO conserva el target, no suponer que falló M2M.
- Analizar toda la colección de candidatos para identificar sustitutos posibles y evitar confundir esencialidad *en el par* con esencialidad *en los 14 modelos*.
- Verificar transporte efectivo de dopamina entre compartimentos/organismos; `mincom` evalúa potencial de producción bajo su abstracción, no un cultivo real.
- Validar posteriormente con balances de masa, restricciones de medio y **FBA/SteadyCom**, distinguiendo factibilidad estacionaria de la capacidad de iniciar dinámicamente un ciclo de transporte.
- Documentar la suposición de NADH y otros cofactores **citosólicos** como seeds, dado que no representan directamente nutrientes extracelulares.

**Advertencia metodológica:** los knockout, el cambio de seed y el transportador hipotético son *intervenciones distintas*; no combinar sus resultados como si fueran condiciones biológicas equivalentes.